# ResNet50 sensitivity analysis without retraining

Public adaptation of the original evaluation notebook. Configure CKPT and DIRS in the first code cell, then run cells in order. Requires Python, torch, torchvision, pandas, numpy, Pillow and scikit-learn. The original images and saved checkpoint must be supplied by the researcher; they are not included here.

The embedded metadata preserve the original 1,496 test records, recorded test-image SHA-256 fingerprints and prior audit pair annotations. The notebook rechecks test-image hashes and restores the saved model without training. It uses prior pair annotations; it does not rerun a complete train/validation duplicate search or SIFT audit.

The manuscript sensitivity scenario is exclude_21_exact_any_overlap (1,475 retained samples). The original 17-, 33- and 40-exclusion scenarios are also preserved for traceability; the latter two include exploratory SIFT candidates, not confirmed exact duplicates. This is post-hoc sensitivity evaluation, not independent external validation.

Architecture, preprocessing, inference and metric calculations are retained from the supplied notebook. Local path configuration, output naming and packaging have been adapted; saved cell outputs and notebook-specific metadata have been removed. Outputs are written to a new directory to avoid mixing runs. Full model inference requires the original images and checkpoint.


In [ ]:
# 1. Configure local inputs and verify fixed test membership.
from pathlib import Path
import io, json, zlib, base64, hashlib, os
import pandas as pd
import numpy as np
from PIL import Image

# Edit these paths before running. Images and checkpoint are not bundled.
CKPT=Path('/path/to/best_resnet50_baseline.pth')
DIRS={1:Path('/path/to/has_arch'),0:Path('/path/to/no_arch')}
OUT=Path('resnet50_sensitivity_output')
OUT.mkdir(parents=True,exist_ok=False)

# Metadata is embedded from the original fixed-test CSV; no second file upload necessary.
EMBEDDED_DATA = """..."""
meta=json.loads(zlib.decompress(base64.b64decode(EMBEDDED_DATA)).decode('utf-8'))
test=pd.DataFrame(meta['test_table'])
assert len(test)==1496 and test.sample_id.is_unique and set(test.label)=={0,1}
assert test.label.value_counts().to_dict()=={1:877,0:619}
assert (test.sample_id==[f'TEST_{i:05d}' for i in range(1,1497)]).all()
assert test.apply(lambda r: r.relative_reference==('has_arch/' if r.label==1 else 'no_arch/')+r.original_filename,axis=1).all()
for folder in DIRS.values():
    if not folder.is_dir():
        raise FileNotFoundError(f'Image folder not found: {folder}')
if not CKPT.is_file():
    raise FileNotFoundError('Original ResNet50 checkpoint not found: '+str(CKPT))

test['image_path']=[str(DIRS[int(r.label)]/r.original_filename) for r in test.itertuples(index=False)]
missing=test.loc[~test.image_path.map(os.path.isfile),['sample_id','image_path']]
if len(missing):
    missing.to_csv(OUT/'MISSING_TEST_PATHS.csv',index=False)
    raise RuntimeError(f'{len(missing)} Test images missing. See '+str(OUT/'MISSING_TEST_PATHS.csv'))

print('Ready: 1496 Test images found, original checkpoint found. No files modified.')


In [ ]:
# 2. Restore exact ResNet50 architecture from the original experiment (INFERENCE ONLY).
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext
import warnings

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE,'Checkpoint:',CKPT)
model=models.resnet50(weights=None)  # Avoid internet download; checkpoint contains the learned weights.
model.fc=nn.Sequential(nn.Dropout(p=0.30),nn.Linear(model.fc.in_features,2))
# The checkpoint was saved by the original project. Loading weights_only avoids arbitrary unpickling.
checkpoint=torch.load(str(CKPT),map_location='cpu',weights_only=True)
state=checkpoint['model_state_dict'] if isinstance(checkpoint,dict) and 'model_state_dict' in checkpoint else checkpoint
model.load_state_dict(state,strict=True)
model=model.to(DEVICE).eval()
TRANSFORM=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
class TestRows(Dataset):
    def __init__(self,tab): self.tab=tab.reset_index(drop=True)
    def __len__(self): return len(self.tab)
    def __getitem__(self,i):
        r=self.tab.iloc[i]
        # One read both provides original pixel bytes and enables independent SHA256 check.
        raw=Path(r.image_path).read_bytes()
        digest=hashlib.sha256(raw).hexdigest()
        with Image.open(io.BytesIO(raw)) as im:
            img=im.convert('RGB')
        return TRANSFORM(img),int(r.label),i,digest
loader=DataLoader(TestRows(test),batch_size=32,shuffle=False,num_workers=0,pin_memory=(DEVICE.type=='cuda'))
print('Model restored; evaluation transform matches original notebook. No optimizer or training loop.')


In [ ]:
# 3. Run inference ONCE, retaining exact sample IDs and original-image SHA-256.
from tqdm.auto import tqdm
records=[]
with torch.inference_mode():
    for x,y,indices,digests in tqdm(loader,desc='Existing model inference'):
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=(DEVICE.type=='cuda')):
            logits=model(x)
        ypred=logits.argmax(dim=1).cpu().numpy()
        probabilities=torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy()
        for row_index,true,pred,prob,sha in zip(indices.numpy(),y.numpy(),ypred,probabilities,digests):
            row=test.iloc[int(row_index)]
            records.append({'sample_id':row.sample_id,'original_filename':row.original_filename,
                            'relative_reference':row.relative_reference,'y_true':int(true),
                            'y_pred':int(pred),'p_has_arch':float(prob),'sha256_current':sha,
                            'sha256_previous_audit':meta['test_audit_sha256'][row.sample_id],
                            'matches_prior_audit_sha256':sha==meta['test_audit_sha256'][row.sample_id]})
preds=pd.DataFrame(records)
assert len(preds)==1496 and preds.sample_id.tolist()==test.sample_id.tolist()
preds['correct']=(preds.y_true==preds.y_pred)
preds.to_csv(OUT/'original_test_predictions.csv',index=False,encoding='utf-8-sig')
num_hash_mismatch=int((~preds.matches_prior_audit_sha256).sum())
print('Predictions saved for',len(preds),'samples.')
print('Images changed since previously recorded audit:',num_hash_mismatch)
if num_hash_mismatch:
    print('WARNING: some current Test image bytes differ from the previous audit. Results may not be comparable.')


In [ ]:
# 4. Compute full-test and tiered sensitivity; NEVER call filtered Test "independent validation".
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from collections import defaultdict

pairdf=pd.DataFrame(meta['pairs'])
assert len(pairdf)==60 and pairdf.pair_id.is_unique

def test_of_pair(p):
    ids=[p.sample_a,p.sample_b]
    t=[x for x in ids if x.startswith('TEST_')]
    if len(t)>1: raise AssertionError('Unexpected test-test pair')
    return t[0] if t else None

def paired_with(p,s):
    other=p.sample_b if p.sample_a==s else p.sample_a
    return other

exact_train=set(); exact_any=set(); high_any=set(); high_train=set()
reasons=defaultdict(list)
for p in pairdf.itertuples(index=False):
    t=test_of_pair(p)
    if not t: continue
    other=paired_with(p,t)
    if p.type=='EXACT_SHA256':
        exact_any.add(t)
        if other.startswith('TRAIN_'): exact_train.add(t)
    else:
        if p.independent_audit_status!='HIGHLY_CONSISTENT_WITH_SAME_IMAGE_OR_DERIVATIVE':
            raise AssertionError('Unexpected high-priority evidence tier')
        high_any.add(t)
        if other.startswith('TRAIN_'): high_train.add(t)
    reasons[t].append({'pair_id':int(p.pair_id),'linked_id':other,'evidence':p.type})

assert (len(exact_train),len(exact_any),len(high_any),len(exact_any|high_any))==(17,21,19,40),(
    len(exact_train),len(exact_any),len(high_any),len(exact_any|high_any))

cases={
 'original_test_all_1496':set(),
 'exclude_17_exact_train_overlap':exact_train,
 'exclude_21_exact_any_overlap':exact_any,
 'exclude_33_exact_or_high_train_overlap':exact_train|high_train,
 'exclude_40_exact_or_high_any_overlap':exact_any|high_any,
}
assert len(cases['exclude_33_exact_or_high_train_overlap'])==33

all_results=[]
for label,excluded in cases.items():
    kept=preds.loc[~preds.sample_id.isin(excluded)]
    matrix=confusion_matrix(kept.y_true,kept.y_pred,labels=[0,1])
    prec,rec,f1,support=precision_recall_fscore_support(kept.y_true,kept.y_pred,labels=[0,1],zero_division=0)
    all_results.append({'scenario':label,'n_kept':len(kept),'n_excluded':len(excluded),
       'accuracy_pct':100*accuracy_score(kept.y_true,kept.y_pred),
       'macro_f1':f1_score(kept.y_true,kept.y_pred,average='macro',labels=[0,1],zero_division=0),
       'no_arch_precision':prec[0],'no_arch_recall':rec[0],'no_arch_f1':f1[0],
       'has_arch_precision':prec[1],'has_arch_recall':rec[1],'has_arch_f1':f1[1],
       'no_arch_support':int(support[0]),'has_arch_support':int(support[1]),
       'tn':int(matrix[0,0]),'fp':int(matrix[0,1]),'fn':int(matrix[1,0]),'tp':int(matrix[1,1])})
results=pd.DataFrame(all_results)
reference_cm=[[560,59],[61,816]]
observed_cm=confusion_matrix(preds.y_true,preds.y_pred,labels=[0,1]).tolist()
reference_reproduced=(observed_cm==reference_cm)
strict_comparable=reference_reproduced and num_hash_mismatch==0
results['original_reference_reproduced']=reference_reproduced
results['all_test_image_hashes_match_prior_audit']=num_hash_mismatch==0
results['comparable_to_published_original']=strict_comparable
results.to_csv(OUT/'SCENARIO_RESULTS.csv',index=False,encoding='utf-8-sig')

exclusion_rows=[]
for s,items in sorted(reasons.items()):
    row=preds.set_index('sample_id').loc[s]
    exclusion_rows.append({'sample_id':s,'y_true':int(row.y_true),'y_pred':int(row.y_pred),
       'correct':bool(row.correct),'excluded_17':s in exact_train,'excluded_21':s in exact_any,
       'excluded_33':s in (exact_train|high_train),'excluded_40':s in (exact_any|high_any),
       'matched_prior_audit_sha256':bool(row.matches_prior_audit_sha256),
       'links':json.dumps(items,ensure_ascii=False)})
pd.DataFrame(exclusion_rows).to_csv(OUT/'EXCLUDED_TEST_SAMPLES_WITH_REASONS.csv',index=False,encoding='utf-8-sig')

summary={'reference_confusion_matrix':reference_cm,'observed_confusion_matrix':observed_cm,
 'reference_reproduced':reference_reproduced,'prior_audit_sha_mismatches':num_hash_mismatch,
 'directly_comparable_to_original':strict_comparable,
 'checkpoint_sha256':hashlib.sha256(CKPT.read_bytes()).hexdigest(),
 'n_rows':len(preds),'embedded_metadata_source_checksums':meta['source_file_sha256'],
 'interpretation':'Post-hoc sensitivity analysis ONLY; no retraining, leakage-free claim, or building-disjoint validation.'}
(OUT/'INTEGRITY_AND_COMPARABILITY.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
print('\nFULL TEST CONFUSION:',observed_cm,'\nExpected:',reference_cm)
print('Audit-matching image SHA-256: ',1496-num_hash_mismatch,'/ 1496')
print('DIRECTLY COMPARABLE TO PUBLISHED ORIGINAL:',strict_comparable)
if not strict_comparable:
    print('IMPORTANT: Original benchmark was NOT reproduced exactly, or image bytes changed; do not interpret filtered numbers as robust evidence for the original publication.')
print(results[['scenario','n_kept','n_excluded','accuracy_pct','macro_f1']].to_string(index=False))


In [ ]:
# 5. Export the two manuscript scenarios using the repository filenames.
import zipfile
retained=preds.loc[~preds.sample_id.isin(exact_any)]
assert len(retained)==1475
retained.to_csv(OUT/'sensitivity_test_predictions.csv',index=False,encoding='utf-8')
results.loc[results.scenario.isin(['original_test_all_1496','exclude_21_exact_any_overlap'])].to_csv(OUT/'sensitivity_results.csv',index=False,encoding='utf-8')
(OUT/'integrity_and_comparability.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
# Include only the four manuscript-related result files in the public archive.
names=['original_test_predictions.csv','sensitivity_test_predictions.csv','sensitivity_results.csv','integrity_and_comparability.json']
zip_path=OUT/'GitHub_Sensitivity_Results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for name in names: z.write(OUT/name,arcname=name)
print('Results archive:',zip_path)
